# Phase 4: Architecture Decision Matrix

In this phase, you act as a Lead Architect and design embedding strategies for five different real-world product scenarios. The goal is to evaluate how model choice changes when constraints such as scale, offline operation, domain precision, multilingual coverage, and multimodal search are the primary drivers.

## Scenario A: Offline Mobile-First Application

**Description:** App works with no internet access

**Model Choice:** `sentence-transformers/all-MiniLM-L6-v2` deployed locally on-device

**Reason:** For a mobile-first app that must work fully offline, the priority is a compact model that can run locally without any dependency on a hosted API. `all-MiniLM-L6-v2` produces 384-dimensional embeddings and is widely used for semantic search, which makes it a practical fit for on-device retrieval.

**Trade offs:** Accuracy will usually be lower than larger hosted embedding models, and device constraints may require quantization, smaller indexes, or lighter retrieval pipelines.

## Scenario B: Large Scale E-Commerce (500M Documents)

**Description:** Strict memory constraints, massive scale

**Model Choice:** `voyage-4-lite` with 256-dimensional output

**Reason:** At 500 million documents, memory efficiency matters as much as retrieval quality. Voyage documents that the 4-series models support reduced output dimensions including 256, and `voyage-4-lite` is optimized for latency and cost, which makes it a strong fit for large-scale indexing.

**Trade offs:** Lower-dimensional embeddings reduce storage and speed up retrieval, but they may lose some ranking fidelity compared with larger vectors or higher-capacity models.

## Scenario C: Legal Discovery Platform

**Description:** Highly specialized legal domain, maximum precision needed

**Model Choice:** `voyage-law-2`

**Reason:** Voyage explicitly recommends `voyage-law-2` for legal retrieval tasks. In a legal discovery setting, domain-specific understanding is more important than minimizing cost, because subtle phrasing and legal terminology directly affect relevance and recall.

**Trade offs:** Domain-tuned legal models can be more expensive and less flexible for unrelated workloads, and they may still benefit from a reranking layer for the highest-stakes search results.

## Scenario D: Global Multilingual SaaS Platform

**Description:** Handles queries in 100+ languages, minimal ML infrastructure

**Model Choice:** `embed-multilingual-v3.0` from Cohere

**Reason:** Cohere documents that `embed-multilingual-v3.0` supports over 100 languages. For a global SaaS platform with minimal ML infrastructure, a managed multilingual embedding API reduces operational burden while still covering a broad language footprint.

**Trade offs:** This keeps the stack simple, but it creates vendor dependency and may not match the absolute best quality available in every single language or domain.

## Scenario E: Multimodal Search System

**Description:** Supports both text based search and image based search

**Model Choice:** `embed-v4.0` from Cohere

**Reason:** Cohere documents that `embed-v4.0` supports text, images, and mixed inputs in a shared embedding space, which is exactly what a multimodal search system needs. It also supports multiple output dimensions, giving room to tune memory and latency.

**Trade offs:** Multimodal systems are more complex to evaluate and tune, and image-capable embeddings can increase cost and operational complexity compared with text-only search.

## Decision Matrix Summary

| Scenario | Model Choice | Key Reason | Main Trade off |
|---|---|---|---|
| Scenario A | `all-MiniLM-L6-v2` | Runs locally without internet | Lower quality than larger hosted models |
| Scenario B | `voyage-4-lite` at 256 dims | Strong scale, low latency, reduced memory | Some accuracy lost from compression |
| Scenario C | `voyage-law-2` | Tuned for legal retrieval | Higher specialization and cost |
| Scenario D | `embed-multilingual-v3.0` | Supports 100+ languages with managed infra | Vendor dependence and uneven language quality |
| Scenario E | `embed-v4.0` | Shared text-image embedding space | More complex and potentially costlier system |

## Analysis Questions

Q1: Which embedding provider/model would you choose for each scenario and why?

A1: For Scenario A, I would choose `sentence-transformers/all-MiniLM-L6-v2` because offline operation matters more than absolute retrieval quality. For Scenario B, I would choose `voyage-4-lite` with 256-dimensional embeddings because the system has to manage huge scale under tight memory limits. For Scenario C, I would choose `voyage-law-2` because legal discovery benefits from a domain-specific model that understands legal phrasing better than a general model. For Scenario D, I would choose Cohere `embed-multilingual-v3.0` because it supports over 100 languages and keeps the infrastructure simple. For Scenario E, I would choose Cohere `embed-v4.0` because it supports a unified embedding space for text and image retrieval.





Q2: What trade offs influenced your decisions across cost, accuracy, latency and infrastructure?

A2: The main trade-off was that the best model is not always the best deployment choice. Offline and mobile scenarios push the design toward smaller local models even if quality drops. Massive-scale systems reward compact embeddings and lower latency because storage and serving costs become dominant. High-stakes legal search shifts the decision toward precision and domain fit, even if inference is more expensive. Global products benefit from managed multilingual APIs because they reduce engineering overhead, while multimodal systems require accepting more architectural complexity in exchange for broader search capability.





Q3: How does a multimodal embedding space differ from a text only embedding space?

A3: A text-only embedding space is designed to represent meaning between language inputs, so similarity is based only on textual semantics. A multimodal embedding space aligns different data types such as text and images into the same vector space, which allows a text query to retrieve an image or an image query to retrieve related text. That makes the representation more flexible, but it also makes evaluation and model design more challenging because the system has to preserve meaning across different modalities.



